# Diagnóstico de calidad de datos · Vehículos eléctricos y PM2.5

**Actividad:** GA1-220501094-01-AA1-EV05 – Diagnosticar la calidad de datos: preparación, exploración y documentación
**Proyecto:** Vehículos eléctricos y calidad del aire: ¿el aumento de carros eléctricos ayuda a reducir el PM2.5 en las ciudades?
**Aprendiz:** Juan Camilo Sarrazola Agudelo · Ficha 3169901 · SENA
**Rol:** analista de datos responsable de evaluar la calidad inicial de los datasets del proyecto

> **Cómo está construido este notebook:** ninguna cifra de las interpretaciones está escrita a mano.
> Cada conteo, porcentaje y hallazgo se calcula con Pandas a partir de los DataFrames cargados, y el
> informe `reports/informe_diagnostico.md` se genera al final con esos mismos resultados. Así el informe
> siempre coincide con los datos que se descargaron.

---
## Momento 1. Reconocimiento del proyecto y de los datasets

### 1.1 Nombre y propósito
**Vehículos eléctricos y calidad del aire.** El proyecto busca revisar con datos si el crecimiento del parque de
vehículos eléctricos se relaciona con una baja del material particulado fino (PM2.5), un contaminante que afecta
la salud respiratoria y cardiovascular.

### 1.2 Pregunta analítica (definida en la EV02)
> ¿El aumento de vehículos eléctricos en las ciudades se relaciona con una baja del PM2.5 (µg/m³) entre 2021 y 2025?

**Pregunta orientadora de esta actividad:** ¿qué condiciones de calidad presentan los conjuntos de datos
seleccionados y qué acciones de preparación o limpieza deberían considerarse antes de iniciar el análisis?

### 1.3 Datasets seleccionados (evaluados en la EV03)

| # | DataFrame | Archivo | Fuente | Qué contiene | Relación con la pregunta |
|---|---|---|---|---|---|
| 1 | `ev_registro` | `Electric_Vehicle_Population_Data.csv` | Data.gov · Departamento de Licencias de Washington | Un registro por vehículo eléctrico (BEV o PHEV) registrado hoy: condado, ciudad, marca, modelo, año, autonomía | Variable principal a nivel local (condado/ciudad) |
| 2 | `ev_historial` | `Electric_Vehicle_Population_Size_History_By_County.csv` | Data.gov · Departamento de Licencias de Washington | Conteo mensual por condado de vehículos eléctricos, no eléctricos y % eléctrico | Da la **evolución en el tiempo** que el registro 1 no tiene (es una foto actual) |
| 3 | `ev_ventas` | `electric-car-sales.csv` | Our World in Data (datos de la IEA) | Autos eléctricos nuevos vendidos por país y año | Contexto internacional de la electrificación |
| 4 | `pm25` | `API_EN.ATM.PM25.MC.M3_DS2_en_csv_v2_….csv` | Banco Mundial · indicador EN.ATM.PM25.MC.M3 | Exposición media anual a PM2.5 (µg/m³) por país, en formato ancho (un año por columna) | Variable que se quiere explicar |

Archivo auxiliar: `Metadata_Country_API_EN.ATM.PM25…csv` (viene en el mismo ZIP del Banco Mundial). Se usa solo para
distinguir países de agregados regionales; no es un dataset del proyecto.

### 1.4 Aspectos de calidad que hay que revisar
1. **Completitud:** valores faltantes en columnas clave (condado, fecha, año, PM2.5).
2. **Tipos de datos:** fechas leídas como texto, códigos (postal, distrito, censo) leídos como números.
3. **Consistencia interna:** totales que deben sumar (BEV + PHEV = total eléctrico), porcentajes recalculables, duplicados.
4. **Valores válidos:** ceros que significan "sin dato", años fuera de rango, valores negativos.
5. **Cobertura:** si los años y lugares de cada dataset cubren el periodo 2021–2025 y si se pueden unir entre sí.

---
## Momento 2. Preparación del entorno y carga de datos

Los CSV se descargan de las fuentes indicadas en el README y se guardan en `data/raw/`. **No se suben a GitHub**
(están en `.gitignore`) porque el registro de vehículos pesa más de 80 MB y se actualiza con frecuencia.

In [1]:
import sys
import warnings
from datetime import date
from pathlib import Path

import pandas as pd
from IPython.display import display, Markdown

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

print("Python", sys.version.split()[0], "| pandas", pd.__version__)


def md(texto):
    """Muestra texto con formato Markdown dentro del notebook."""
    display(Markdown(texto))


def fmt(n):
    """Entero con punto de miles: 254371 -> 254.371"""
    return f"{n:,.0f}".replace(",", ".")


def pct(x, dec=2):
    """Porcentaje con coma decimal: 12.345 -> 12,35 %"""
    return f"{x:,.{dec}f}".replace(",", "X").replace(".", ",").replace("X", ".") + " %"


def valor(v, clave=""):
    """Formato para tablas: enteros con punto de miles (salvo años), el resto como texto."""
    if "year" in clave.lower() or "año" in clave.lower():
        return str(v)
    if isinstance(v, bool):
        return "sí" if v else "no"
    if isinstance(v, (int,)) or (isinstance(v, float) and v.is_integer() and abs(v) >= 1000):
        return fmt(v)
    return str(v)


def tabla_md(df):
    """Convierte un DataFrame en tabla Markdown (sin dependencias extra)."""
    cols = [str(c) for c in df.columns]
    lineas = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
    for fila in df.itertuples(index=False):
        celdas = [str(v).replace("|", "\\|").replace("\n", " ") for v in fila]
        lineas.append("| " + " | ".join(celdas) + " |")
    return "\n".join(lineas)

Python 3.14.3 | pandas 3.1.0


### 2.1 Carga de cada dataset en un DataFrame independiente
* Los archivos de Data.gov y Our World in Data son CSV estándar (separador `,`).
* El CSV del Banco Mundial trae **4 líneas de encabezado** antes de los nombres de columna, por eso se usa `skiprows=4`.
* `low_memory=False` hace que Pandas lea cada columna completa antes de decidir su tipo (evita tipos mezclados en archivos grandes).

In [2]:
RUTA_DATOS = Path("../data/raw")
if not RUTA_DATOS.exists():
    RUTA_DATOS = Path("data/raw")


def ruta(nombre, patron=None):
    """Devuelve la ruta del archivo; si se descargó a mano, lo busca por patrón."""
    p = RUTA_DATOS / nombre
    if p.exists():
        return p
    if patron:
        encontrados = sorted(RUTA_DATOS.glob(patron))
        if encontrados:
            return encontrados[0]
    raise FileNotFoundError(
        f"No se encontró {nombre} en {RUTA_DATOS.resolve()}. Copie los CSV en data/raw (ver README)."
    )


ev_registro = pd.read_csv(ruta("Electric_Vehicle_Population_Data.csv"), low_memory=False)
ev_historial = pd.read_csv(ruta("Electric_Vehicle_Population_Size_History_By_County.csv"), low_memory=False)
ev_ventas = pd.read_csv(ruta("electric-car-sales.csv"))
RUTA_PM25 = ruta("pm25_banco_mundial.csv", "API_EN.ATM.PM25*.csv")
pm25 = pd.read_csv(RUTA_PM25, skiprows=4)
pm25_paises = pd.read_csv(ruta("pm25_metadatos_paises.csv", "Metadata_Country_API_EN.ATM.PM25*.csv"))

dataframes = {
    "ev_registro": ev_registro,
    "ev_historial": ev_historial,
    "ev_ventas": ev_ventas,
    "pm25": pm25,
}
archivos = {
    "ev_registro": "Electric_Vehicle_Population_Data.csv",
    "ev_historial": "Electric_Vehicle_Population_Size_History_By_County.csv",
    "ev_ventas": "electric-car-sales.csv",
    "pm25": RUTA_PM25.name,
}

### 2.2 Verificación de la carga

In [3]:
verificacion = pd.DataFrame(
    [
        {"DataFrame": n, "archivo": archivos[n], "filas": df.shape[0], "columnas": df.shape[1],
         "carga correcta": "sí" if df.shape[0] > 0 and df.shape[1] > 1 else "NO"}
        for n, df in dataframes.items()
    ]
)
display(verificacion)
assert (verificacion["carga correcta"] == "sí").all(), "Algún archivo quedó vacío o mal leído"
md(f"Los **{len(dataframes)}** datasets se cargaron correctamente. "
   f"Archivo auxiliar `pm25_paises`: {fmt(len(pm25_paises))} filas.")

,DataFrame,archivo,filas,columnas,carga correcta
0,ev_registro,Electric_Vehicle_Population_Data.csv,300142,16,sí
1,ev_historial,Electric_Vehicle_Population_Size_History_By_Co...,36913,10,sí
2,ev_ventas,electric-car-sales.csv,823,4,sí
3,pm25,API_EN.ATM.PM25.MC.M3_DS2_en_csv_v2_648.csv,266,71,sí


Los **4** datasets se cargaron correctamente. Archivo auxiliar `pm25_paises`: 265 filas.

---
## Momento 3. Inspección de la estructura de los datasets
Para cada DataFrame se ejecutan, en este orden: `head()`, `tail()`, `shape`, `columns`, `dtypes`, `info()`,
`describe()` y `describe(include='object')`.

> **Nota sobre pandas 3:** desde pandas 3.0 las columnas de texto tienen el tipo `str` en lugar de `object`.
> `describe(include='object')` las sigue incluyendo (por compatibilidad) pero muestra un aviso de obsolescencia;
> por eso se ejecuta dentro de `warnings.catch_warnings()`.

### Dataset 1 · `ev_registro` (vehículos eléctricos registrados en Washington)

**`head()`** – primeros registros

In [4]:
ev_registro.head()

,VIN (1-10),County,City,State,Postal Code,Model Year,Make,Model,Electric Vehicle Type,Clean Alternative Fuel Vehicle (CAFV) Eligibility,Electric Range,Legislative District,DOL Vehicle ID,Vehicle Location,Electric Utility,2020 GEOID
0,3FMTK3SU8P,King,Seattle,WA,98177.0,2023,FORD,MUSTANG MACH-E,Battery Electric Vehicle (BEV),Eligibility unknown as battery range has not b...,0.0,32.0,254622924,POINT (-122.37238 47.74068),CITY OF SEATTLE - (WA)|CITY OF TACOMA - (WA),5.303300e+10
1,JM3KKEHA8S,Thurston,Yelm,WA,98597.0,2025,MAZDA,CX-90,Plug-in Hybrid Electric Vehicle (PHEV),Not eligible due to low battery range,26.0,2.0,277791643,POINT (-122.60735 46.94239),PUGET SOUND ENERGY INC,5.306701e+10
2,5YJ3E1EB8J,King,Sammamish,WA,98075.0,2018,TESLA,MODEL 3,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,215.0,41.0,347418676,POINT (-122.03539 47.61344),PUGET SOUND ENERGY INC||CITY OF TACOMA - (WA),5.303303e+10
3,JHMZC5F12K,Island,Oak Harbor,WA,98277.0,2019,HONDA,CLARITY,Plug-in Hybrid Electric Vehicle (PHEV),Clean Alternative Fuel Vehicle Eligible,47.0,10.0,223620701,POINT (-122.64682 48.29077),PUGET SOUND ENERGY INC,5.302997e+10
4,1C4RJXN65R,Kitsap,Bremerton,WA,98312.0,2024,JEEP,WRANGLER,Plug-in Hybrid Electric Vehicle (PHEV),Not eligible due to low battery range,21.0,26.0,256402674,POINT (-122.65223 47.57192),PUGET SOUND ENERGY INC,5.303508e+10


**`tail()`** – últimos registros

In [5]:
ev_registro.tail()

,VIN (1-10),County,City,State,Postal Code,Model Year,Make,Model,Electric Vehicle Type,Clean Alternative Fuel Vehicle (CAFV) Eligibility,Electric Range,Legislative District,DOL Vehicle ID,Vehicle Location,Electric Utility,2020 GEOID
300137,5YJ3E1EA6T,King,Kent,WA,98031.0,2026,TESLA,MODEL 3,Battery Electric Vehicle (BEV),Eligibility unknown as battery range has not b...,0.0,47.0,292759134,POINT (-122.20125 47.39318),PUGET SOUND ENERGY INC||CITY OF TACOMA - (WA),5.303303e+10
300138,5YJ3E1EA4R,Franklin,Pasco,WA,99301.0,2024,TESLA,MODEL 3,Battery Electric Vehicle (BEV),Eligibility unknown as battery range has not b...,0.0,9.0,272852555,POINT (-119.09467 46.23542),BONNEVILLE POWER ADMINISTRATION||PUD NO 1 OF F...,5.302102e+10
300139,5YJ3E1EC5M,Snohomish,Snohomish,WA,98296.0,2021,TESLA,MODEL 3,Battery Electric Vehicle (BEV),Eligibility unknown as battery range has not b...,0.0,44.0,166817724,POINT (-122.15134 47.88512),PUGET SOUND ENERGY INC,5.306105e+10
300140,1C4JJXR67P,Stevens,Loon Lake,WA,99148.0,2023,JEEP,WRANGLER,Plug-in Hybrid Electric Vehicle (PHEV),Not eligible due to low battery range,21.0,7.0,251080987,POINT (-117.63731 48.06288),NO KNOWN ELECTRIC UTILITY SERVICE,5.306595e+10
300141,1C4RJYB67P,Spokane,Deer Park,WA,99006.0,2023,JEEP,GRAND CHEROKEE,Plug-in Hybrid Electric Vehicle (PHEV),Not eligible due to low battery range,25.0,7.0,256123444,POINT (-117.47154 47.95431),BONNEVILLE POWER ADMINISTRATION||AVISTA CORP||...,5.306301e+10


**`shape`** – filas y columnas

In [6]:
ev_registro.shape

(300142, 16)

**`columns`** – nombres de las variables

In [7]:
ev_registro.columns

Index(['VIN (1-10)', 'County', 'City', 'State', 'Postal Code', 'Model Year', 'Make', 'Model', 'Electric Vehicle Type', 'Clean Alternative Fuel Vehicle (CAFV) Eligibility', 'Electric Range',
       'Legislative District', 'DOL Vehicle ID', 'Vehicle Location', 'Electric Utility', '2020 GEOID'],
      dtype='str')

**`dtypes`** – tipo de cada variable

In [8]:
ev_registro.dtypes

VIN (1-10)                                               str
County                                                   str
City                                                     str
State                                                    str
Postal Code                                          float64
Model Year                                             int64
Make                                                     str
Model                                                    str
Electric Vehicle Type                                    str
Clean Alternative Fuel Vehicle (CAFV) Eligibility        str
Electric Range                                       float64
Legislative District                                 float64
DOL Vehicle ID                                         int64
Vehicle Location                                         str
Electric Utility                                         str
2020 GEOID                                           float64
dtype: object

**`info()`** – estructura y valores no nulos

In [9]:
ev_registro.info()

<class 'pandas.DataFrame'>
RangeIndex: 300142 entries, 0 to 300141
Data columns (total 16 columns):
 #   Column                                             Non-Null Count   Dtype  
---  ------                                             --------------   -----  
 0   VIN (1-10)                                         300142 non-null  str    
 1   County                                             300127 non-null  str    
 2   City                                               300127 non-null  str    
 3   State                                              300142 non-null  str    
 4   Postal Code                                        300127 non-null  float64
 5   Model Year                                         300142 non-null  int64  
 6   Make                                               300142 non-null  str    
 7   Model                                              300142 non-null  str    
 8   Electric Vehicle Type                              300142 non-null  str    
 9   Clea

**`describe()`** – estadísticas de las variables numéricas

In [10]:
ev_registro.describe()

,Postal Code,Model Year,Electric Range,Legislative District,DOL Vehicle ID,2020 GEOID
count,300127.000000,300142.000000,300107.000000,299352.000000,3.001420e+05,3.001270e+05
mean,98175.788819,2022.389056,36.148437,28.719230,2.516250e+08,5.296688e+10
std,2669.566595,3.095011,75.443498,14.886897,6.211185e+07,1.696036e+09
min,1030.000000,1999.000000,0.000000,1.000000,4.385000e+03,1.001020e+09
25%,98052.000000,2021.000000,0.000000,17.000000,2.287301e+08,5.303301e+10
50%,98134.000000,2023.000000,0.000000,32.000000,2.686437e+08,5.303303e+10
75%,98391.000000,2025.000000,32.000000,42.000000,2.851263e+08,5.305394e+10
max,99403.000000,2027.000000,337.000000,49.000000,4.791150e+08,6.601095e+10


**`describe(include='object')`** – variables de texto

In [11]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    display(ev_registro.describe(include="object"))

,VIN (1-10),County,City,State,Make,Model,Electric Vehicle Type,Clean Alternative Fuel Vehicle (CAFV) Eligibility,Vehicle Location,Electric Utility
count,300142,300127,300127,300142,300142,300142,300142,300142,300118,300127
unique,18526,254,934,51,52,205,2,3,1197,76
top,7SAYGDEE9T,King,Seattle,WA,TESLA,MODEL Y,Battery Electric Vehicle (BEV),Eligibility unknown as battery range has not b...,POINT (-122.13158 47.67858),PUGET SOUND ENERGY INC||CITY OF TACOMA - (WA)
freq,1229,143976,45446,299353,122291,66584,241840,197085,7118,103771


### Dataset 2 · `ev_historial` (historial mensual por condado)

**`head()`** – primeros registros

In [12]:
ev_historial.head()

,Date,County,State,Vehicle Primary Use,Battery Electric Vehicles (BEVs),Plug-In Hybrid Electric Vehicles (PHEVs),Electric Vehicle (EV) Total,Non-Electric Vehicle Total,Total Vehicles,Percent Electric Vehicles
0,August 31 2023,Ada,ID,Passenger,4,0,4,218,222,1.80
1,April 30 2023,Lee,AL,Passenger,1,1,2,23,25,8.00
2,November 30 2024,Hendricks,IN,Passenger,1,0,1,10,11,9.09
3,December 31 2021,Wahkiakum,WA,Truck,0,0,0,1820,1820,0.00
4,April 30 2024,Essex,MA,Passenger,2,0,2,16,18,11.11


**`tail()`** – últimos registros

In [13]:
ev_historial.tail()

,Date,County,State,Vehicle Primary Use,Battery Electric Vehicles (BEVs),Plug-In Hybrid Electric Vehicles (PHEVs),Electric Vehicle (EV) Total,Non-Electric Vehicle Total,Total Vehicles,Percent Electric Vehicles
36908,October 31 2025,Pottawatomie,KS,Passenger,1,0,1,6,7,14.29
36909,December 31 2022,Chatham,NC,Passenger,1,0,1,9,10,10.00
36910,June 30 2023,Kent,DE,Passenger,1,0,1,53,54,1.85
36911,March 31 2022,Maui,HI,Passenger,2,0,2,29,31,6.45
36912,June 30 2024,Sussex,DE,Passenger,1,0,1,13,14,7.14


**`shape`** – filas y columnas

In [14]:
ev_historial.shape

(36913, 10)

**`columns`** – nombres de las variables

In [15]:
ev_historial.columns

Index(['Date', 'County', 'State', 'Vehicle Primary Use', 'Battery Electric Vehicles (BEVs)', 'Plug-In Hybrid Electric Vehicles (PHEVs)', 'Electric Vehicle (EV) Total', 'Non-Electric Vehicle Total',
       'Total Vehicles', 'Percent Electric Vehicles'],
      dtype='str')

**`dtypes`** – tipo de cada variable

In [16]:
ev_historial.dtypes

Date                                            str
County                                          str
State                                           str
Vehicle Primary Use                             str
Battery Electric Vehicles (BEVs)              int64
Plug-In Hybrid Electric Vehicles (PHEVs)      int64
Electric Vehicle (EV) Total                   int64
Non-Electric Vehicle Total                    int64
Total Vehicles                                int64
Percent Electric Vehicles                   float64
dtype: object

**`info()`** – estructura y valores no nulos

In [17]:
ev_historial.info()

<class 'pandas.DataFrame'>
RangeIndex: 36913 entries, 0 to 36912
Data columns (total 10 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   Date                                      36913 non-null  str    
 1   County                                    36796 non-null  str    
 2   State                                     36796 non-null  str    
 3   Vehicle Primary Use                       36913 non-null  str    
 4   Battery Electric Vehicles (BEVs)          36913 non-null  int64  
 5   Plug-In Hybrid Electric Vehicles (PHEVs)  36913 non-null  int64  
 6   Electric Vehicle (EV) Total               36913 non-null  int64  
 7   Non-Electric Vehicle Total                36913 non-null  int64  
 8   Total Vehicles                            36913 non-null  int64  
 9   Percent Electric Vehicles                 36913 non-null  float64
dtypes: float64(1), int64(5), str(4)
memory usage:

**`describe()`** – estadísticas de las variables numéricas

In [18]:
ev_historial.describe()

,Battery Electric Vehicles (BEVs),Plug-In Hybrid Electric Vehicles (PHEVs),Electric Vehicle (EV) Total,Non-Electric Vehicle Total,Total Vehicles,Percent Electric Vehicles
count,36913.000000,36913.000000,36913.000000,3.691300e+04,3.691300e+04,36913.000000
mean,287.823179,88.189012,376.012191,1.913969e+04,1.951571e+04,6.063194
std,3474.752631,855.827992,4321.304533,9.299251e+04,9.627375e+04,13.096507
min,0.000000,0.000000,0.000000,0.000000e+00,1.000000e+00,0.000000
25%,1.000000,0.000000,1.000000,2.700000e+01,2.900000e+01,0.740000
50%,1.000000,0.000000,2.000000,9.600000e+01,9.800000e+01,2.240000
75%,4.000000,1.000000,5.000000,1.248000e+03,1.262000e+03,5.450000
max,114412.000000,25192.000000,139071.000000,1.397720e+06,1.429685e+06,100.000000


**`describe(include='object')`** – variables de texto

In [19]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    display(ev_historial.describe(include="object"))

,Date,County,State,Vehicle Primary Use
count,36913,36796,36796,36913
unique,117,422,51,2
top,November 30 2024,Jefferson,WA,Passenger
freq,430,541,9126,31365


### Dataset 3 · `ev_ventas` (ventas de autos eléctricos por país)

**`head()`** – primeros registros

In [20]:
ev_ventas.head()

,Entity,Code,Year,Electric cars sold
0,Australia,AUS,2011,49
1,Australia,AUS,2012,250
2,Australia,AUS,2013,290
3,Australia,AUS,2014,1320
4,Australia,AUS,2015,1760


**`tail()`** – últimos registros

In [21]:
ev_ventas.tail()

,Entity,Code,Year,Electric cars sold
818,World,OWID_WRL,2021,6600000
819,World,OWID_WRL,2022,10200000
820,World,OWID_WRL,2023,13700000
821,World,OWID_WRL,2024,17500000
822,World,OWID_WRL,2025,21200000


**`shape`** – filas y columnas

In [22]:
ev_ventas.shape

(823, 4)

**`columns`** – nombres de las variables

In [23]:
ev_ventas.columns

Index(['Entity', 'Code', 'Year', 'Electric cars sold'], dtype='str')

**`dtypes`** – tipo de cada variable

In [24]:
ev_ventas.dtypes

Entity                  str
Code                    str
Year                  int64
Electric cars sold    int64
dtype: object

**`info()`** – estructura y valores no nulos

In [25]:
ev_ventas.info()

<class 'pandas.DataFrame'>
RangeIndex: 823 entries, 0 to 822
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   Entity              823 non-null    str  
 1   Code                807 non-null    str  
 2   Year                823 non-null    int64
 3   Electric cars sold  823 non-null    int64
dtypes: int64(2), str(2)
memory usage: 25.8 KB


**`describe()`** – estadísticas de las variables numéricas

In [26]:
ev_ventas.describe()

,Year,Electric cars sold
count,823.000000,8.230000e+02
mean,2018.533414,2.326142e+05
std,4.328769,1.388813e+06
min,2010.000000,-4.975200e+04
25%,2015.000000,4.100000e+02
50%,2019.000000,4.700000e+03
75%,2022.000000,4.200000e+04
max,2025.000000,2.120000e+07


**`describe(include='object')`** – variables de texto

In [27]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    display(ev_ventas.describe(include="object"))

,Entity,Code
count,823,807
unique,64,63
top,Belgium,BEL
freq,16,16


### Dataset 4 · `pm25` (exposición media anual a PM2.5 por país)

**`head()`** – primeros registros

In [28]:
pm25.head()

,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,1966,1967,1968,1969,1970,1971,1972,1973,1974,1975,1976,1977,1978,1979,1980,1981,1982,1983,1984,1985,1986,1987,1988,1989,1990,1991,1992,1993,1994,1995,1996,1997,1998,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,Unnamed: 70
0,Aruba,ABW,"PM2.5 air pollution, mean annual exposure (mic...",EN.ATM.PM25.MC.M3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Africa Eastern and Southern,AFE,"PM2.5 air pollution, mean annual exposure (mic...",EN.ATM.PM25.MC.M3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,23.793024,23.392693,23.536270,23.674895,23.772252,22.797438,24.043646,24.193431,24.337854,24.485975,25.184790,27.060941,25.642607,24.701003,25.216018,25.115747,25.652229,26.268869,26.685112,25.866904,28.179055,27.443605,27.328477,27.710998,24.567837,26.734817,25.671186,24.786789,24.875242,26.282024,25.660129,25.529458,26.383916,25.283403,NaN,NaN,NaN
2,Afghanistan,AFG,"PM2.5 air pollution, mean annual exposure (mic...",EN.ATM.PM25.MC.M3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,48.426586,47.941609,47.974404,48.007198,48.039993,47.037244,48.105582,48.138377,48.171171,48.203966,48.754532,47.343913,49.618784,48.647668,47.392007,48.098084,40.538015,37.727037,36.852263,34.591442,39.088230,41.902605,34.349226,36.612810,35.981158,36.485734,36.872527,37.066798,40.838489,34.795214,34.164126,39.738833,40.206779,37.948311,NaN,NaN,NaN
3,Africa Western and Central,AFW,"PM2.5 air pollution, mean annual exposure (mic...",EN.ATM.PM25.MC.M3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,51.095358,49.988480,50.224302,50.465567,50.716583,48.193560,51.196527,51.420409,51.637512,51.875697,53.505915,45.677270,49.113554,42.070396,51.926816,46.697182,50.287657,53.168573,53.226260,48.236506,46.625610,49.172025,48.607575,42.615884,43.797674,58.855574,54.331008,53.058490,47.264084,44.582794,49.711336,49.335972,53.622398,50.424014,NaN,NaN,NaN
4,Angola,AGO,"PM2.5 air pollution, mean annual exposure (mic...",EN.ATM.PM25.MC.M3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,23.676964,23.367735,23.344756,23.321777,23.298798,22.703318,23.252839,23.229860,23.206881,23.183902,23.447173,24.313643,24.056663,23.431801,23.372055,24.275362,20.910536,20.304698,21.384628,20.217795,24.585570,21.523624,21.343442,22.642942,19.356391,21.232451,20.996796,18.226159,19.927565,22.174344,19.570648,18.490841,22.517224,19.907569,NaN,NaN,NaN


**`tail()`** – últimos registros

In [29]:
pm25.tail()

,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,1966,1967,1968,1969,1970,1971,1972,1973,1974,1975,1976,1977,1978,1979,1980,1981,1982,1983,1984,1985,1986,1987,1988,1989,1990,1991,1992,1993,1994,1995,1996,1997,1998,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,Unnamed: 70
261,Kosovo,XKX,"PM2.5 air pollution, mean annual exposure (mic...",EN.ATM.PM25.MC.M3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
262,"Yemen, Rep.",YEM,"PM2.5 air pollution, mean annual exposure (mic...",EN.ATM.PM25.MC.M3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,44.764174,42.939473,43.076223,43.212973,43.349723,39.563571,43.623223,43.759973,43.896723,44.033473,46.131675,38.243771,37.045857,39.127572,38.989895,39.768360,39.473310,40.800011,41.655268,41.003143,41.211699,48.376699,53.858465,51.870421,42.725382,50.050350,46.077933,47.973257,45.663183,48.335147,43.734771,41.734805,47.114649,46.835978,NaN,NaN,NaN
263,South Africa,ZAF,"PM2.5 air pollution, mean annual exposure (mic...",EN.ATM.PM25.MC.M3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,23.681032,23.877188,23.870336,23.863483,23.856631,24.255796,23.842926,23.836074,23.829222,23.822369,23.612508,22.964682,22.377216,21.299676,21.448330,21.702947,25.785000,27.936366,27.974333,28.401135,32.795927,28.534440,28.647488,28.406424,26.749291,28.496468,26.143646,28.757631,27.773423,28.869769,28.326217,27.722593,28.506282,27.724711,NaN,NaN,NaN
264,Zambia,ZMB,"PM2.5 air pollution, mean annual exposure (mic...",EN.ATM.PM25.MC.M3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,21.728272,21.288460,21.405178,21.521895,21.638613,20.642272,21.872048,21.988765,22.105483,22.222200,22.895447,25.467707,23.107461,22.375571,21.640604,22.679974,22.702128,23.254075,23.808937,23.069895,24.476529,22.477051,22.998331,22.081612,20.533931,22.761735,21.213419,19.815295,21.114438,22.120533,19.448496,19.592501,23.490185,20.243049,NaN,NaN,NaN
265,Zimbabwe,ZWE,"PM2.5 air pollution, mean annual exposure (mic...",EN.ATM.PM25.MC.M3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16.217468,16.004938,16.025957,16.046977,16.067996,15.621917,16.110035,16.131054,16.152074,16.173093,16.427662,20.541919,17.683773,15.812917,14.766578,16.625577,18.488492,19.511356,19.597049,17.942404,20.295026,16.765481,18.374980,17.255819,15.275086,17.825025,15.692757,15.598419,18.027732,16.533665,13.698771,15.144709,17.390179,15.025473,NaN,NaN,NaN


**`shape`** – filas y columnas

In [30]:
pm25.shape

(266, 71)

**`columns`** – nombres de las variables

In [31]:
pm25.columns

Index(['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code', '1960', '1961', '1962', '1963', '1964', '1965', '1966', '1967', '1968', '1969', '1970', '1971', '1972', '1973', '1974',
       '1975', '1976', '1977', '1978', '1979', '1980', '1981', '1982', '1983', '1984', '1985', '1986', '1987', '1988', '1989', '1990', '1991', '1992', '1993', '1994', '1995', '1996', '1997', '1998',
       '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022',
       '2023', '2024', '2025', 'Unnamed: 70'],
      dtype='str')

**`dtypes`** – tipo de cada variable

In [32]:
pm25.dtypes

Country Name          str
Country Code          str
Indicator Name        str
Indicator Code        str
1960              float64
                   ...   
2022              float64
2023              float64
2024              float64
2025              float64
Unnamed: 70       float64
Length: 71, dtype: object

**`info()`** – estructura y valores no nulos

In [33]:
pm25.info()

<class 'pandas.DataFrame'>
RangeIndex: 266 entries, 0 to 265
Data columns (total 71 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Country Name    266 non-null    str    
 1   Country Code    266 non-null    str    
 2   Indicator Name  266 non-null    str    
 3   Indicator Code  266 non-null    str    
 4   1960            0 non-null      float64
 5   1961            0 non-null      float64
 6   1962            0 non-null      float64
 7   1963            0 non-null      float64
 8   1964            0 non-null      float64
 9   1965            0 non-null      float64
 10  1966            0 non-null      float64
 11  1967            0 non-null      float64
 12  1968            0 non-null      float64
 13  1969            0 non-null      float64
 14  1970            0 non-null      float64
 15  1971            0 non-null      float64
 16  1972            0 non-null      float64
 17  1973            0 non-null      float64
 18  1

**`describe()`** – estadísticas de las variables numéricas

In [34]:
pm25.describe()

,1960,1961,1962,1963,1964,1965,1966,1967,1968,1969,1970,1971,1972,1973,1974,1975,1976,1977,1978,1979,1980,1981,1982,1983,1984,1985,1986,1987,1988,1989,1990,1991,1992,1993,1994,1995,1996,1997,1998,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,Unnamed: 70
count,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,248.000000,248.000000,248.000000,248.000000,248.000000,248.000000,248.000000,248.000000,248.000000,248.000000,248.000000,248.000000,248.000000,248.000000,248.000000,248.000000,248.000000,248.000000,248.000000,248.000000,248.000000,248.000000,248.000000,248.000000,248.000000,248.000000,248.000000,248.000000,248.000000,248.000000,248.000000,248.000000,248.000000,248.000000,0.0,0.0,0.0
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,27.821292,27.476376,27.432642,27.388890,27.344383,26.696562,27.256635,27.213447,27.170314,27.128051,27.388097,26.412588,26.850123,26.761571,26.601383,26.315110,25.365885,25.123498,25.311831,24.949206,25.352489,25.509968,25.045158,24.314817,23.492292,25.450263,24.222581,24.136068,24.023305,23.289248,23.043269,23.463019,24.025981,23.204401,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15.269459,15.008295,14.964000,14.926062,14.893560,14.476539,14.846660,14.832402,14.824305,14.822847,15.090034,14.312685,14.265313,15.221551,15.714682,15.236874,14.353432,14.640238,15.078049,15.004281,14.921287,15.311610,15.690394,14.841953,14.287974,16.500545,15.802402,15.680979,15.664284,14.687590,14.991038,15.605278,16.195722,15.672117,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.652726,1.648215,1.636196,1.624177,1.612158,1.615153,1.588120,1.576100,1.564081,1.552062,1.532536,1.552508,1.458529,1.519786,1.386852,1.401449,1.701962,1.862867,1.914730,1.926461,1.980986,1.825683,2.084279,1.820589,1.927912,1.647585,1.582053,1.736211,1.506005,1.846401,1.711347,1.720654,1.773278,1.636285,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16.708912,16.251274,16.267344,16.288942,16.304178,15.911455,16.653449,17.063011,17.250000,17.322078,17.190993,16.735639,16.778687,16.436186,15.644572,16.289759,15.477294,14.948558,15.130177,15.168240,15.139261,15.034529,14.032305,14.070832,13.074681,13.534045,13.086058,13.108215,12.958080,12.400421,11.737031,12.365708,12.169265,11.500417,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,25.756639,26.328687,25.980137,25.810857,25.643314,25.207218,25.060022,24.959188,24.986143,24.887177,24.867803,24.450005,25.139118,25.268856,23.465021,23.977000,22.078046,21.156364,21.133209,20.749344,21.510723,21.494552,20.729913,20.300259,19.741650,20.618687,19.646757,19.206315,19.782833,19.511537,19.243284,18.985264,19.657336,18.961442,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,37.465810,37.427100,37.312464,37.075207,36.825689,36.031872,37.034636,37.080111,36.943739,36.978452,37.373495,35.952092,36.688100,35.464266,37.531992,36.393025,35.580765,35.127966,36.339735,34.685533,35.687207,36.017430,35.124044,33.784742,32.425368,36.498274,35.491079,33.153030,33.345805,33.108599,32.733256,32.657353,32.851158,32.666852,NaN,NaN,NaN
max,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,109.799288,107.804871,107.764685,107.724499,107.684313,103.735665,107.603942,107.563756,107.523570,107.483384,109.397430,104.611839,94.059113,124.657449,117.675164,120.054910,97.025919,90.606125,96.

**`describe(include='object')`** – variables de texto

In [35]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    display(pm25.describe(include="object"))

,Country Name,Country Code,Indicator Name,Indicator Code
count,266,266,266,266
unique,266,266,1,1
top,Aruba,ABW,"PM2.5 air pollution, mean annual exposure (mic...",EN.ATM.PM25.MC.M3
freq,1,1,266,266


### 3.1 Características principales de cada dataset
Resumen calculado a partir de los comandos anteriores.

In [36]:
QUE_ES_UNA_FILA = {
    "ev_registro": "un vehículo eléctrico registrado actualmente en el Departamento de Licencias de Washington",
    "ev_historial": "un mes, un condado y un uso del vehículo (pasajeros o camión)",
    "ev_ventas": "un país o región en un año",
    "pm25": "un país o agregado regional; cada año es una columna (formato ancho)",
}


def tipos_resumen(df):
    conteo = df.dtypes.astype(str).value_counts()
    return ", ".join(f"{t}: {c}" for t, c in conteo.items())


for nombre, df in dataframes.items():
    num = df.select_dtypes("number").shape[1]
    txt = df.shape[1] - num
    cols_nulas = int((df.isna().sum() > 0).sum())
    md(f"""
**{nombre}** · cada fila representa {QUE_ES_UNA_FILA[nombre]}.
- {fmt(df.shape[0])} filas y {df.shape[1]} columnas ({num} numéricas, {txt} de texto u otro tipo).
- Tipos: {tipos_resumen(df)}.
- Columnas con al menos un valor faltante: {cols_nulas} de {df.shape[1]}.
- Filas completamente duplicadas: {fmt(df.duplicated().sum())}.
""")


**ev_registro** · cada fila representa un vehículo eléctrico registrado actualmente en el Departamento de Licencias de Washington.
- 300.142 filas y 16 columnas (6 numéricas, 10 de texto u otro tipo).
- Tipos: str: 10, float64: 4, int64: 2.
- Columnas con al menos un valor faltante: 8 de 16.
- Filas completamente duplicadas: 0.



**ev_historial** · cada fila representa un mes, un condado y un uso del vehículo (pasajeros o camión).
- 36.913 filas y 10 columnas (6 numéricas, 4 de texto u otro tipo).
- Tipos: int64: 5, str: 4, float64: 1.
- Columnas con al menos un valor faltante: 2 de 10.
- Filas completamente duplicadas: 0.



**ev_ventas** · cada fila representa un país o región en un año.
- 823 filas y 4 columnas (2 numéricas, 2 de texto u otro tipo).
- Tipos: str: 2, int64: 2.
- Columnas con al menos un valor faltante: 1 de 4.
- Filas completamente duplicadas: 0.



**pm25** · cada fila representa un país o agregado regional; cada año es una columna (formato ancho).
- 266 filas y 71 columnas (67 numéricas, 4 de texto u otro tipo).
- Tipos: float64: 67, str: 4.
- Columnas con al menos un valor faltante: 67 de 71.
- Filas completamente duplicadas: 0.


### 3.2 Tabla de diagnóstico estructural

In [37]:
OBS_EXTRA = {
    "ev_registro": "Foto actual del parque (no es una serie de tiempo).",
    "ev_historial": "Serie mensual; la fecha viene como texto.",
    "ev_ventas": "Mezcla países y agregados (mundo, regiones).",
    "pm25": "Formato ancho: un año por columna; incluye agregados regionales.",
}

estructura = pd.DataFrame(
    [
        {
            "Dataset": nombre,
            "Filas": fmt(df.shape[0]),
            "Columnas": df.shape[1],
            "Tipos de datos": tipos_resumen(df),
            "Observaciones": (
                f"{int((df.isna().sum() > 0).sum())} columnas con faltantes; "
                f"{fmt(df.duplicated().sum())} filas duplicadas. {OBS_EXTRA[nombre]}"
            ),
        }
        for nombre, df in dataframes.items()
    ]
)
estructura

,Dataset,Filas,Columnas,Tipos de datos,Observaciones
0,ev_registro,300.142,16,"str: 10, float64: 4, int64: 2",8 columnas con faltantes; 0 filas duplicadas. ...
1,ev_historial,36.913,10,"int64: 5, str: 4, float64: 1",2 columnas con faltantes; 0 filas duplicadas. ...
2,ev_ventas,823,4,"str: 2, int64: 2",1 columnas con faltantes; 0 filas duplicadas. ...
3,pm25,266,71,"float64: 67, str: 4",67 columnas con faltantes; 0 filas duplicadas....


---
## Momento 4. Diagnóstico inicial de calidad
En esta fase **no se corrige ni se elimina ningún registro**: solo se detectan, miden y documentan los problemas.
Las comprobaciones son defensivas: si una columna cambió de nombre en una descarga futura, se informa en lugar de fallar.

### 4.1 Valores faltantes

In [38]:
filas_nulos = []
for nombre, df in dataframes.items():
    for col, n in df.isna().sum().items():
        if n > 0:
            filas_nulos.append({"Dataset": nombre, "columna": col, "faltantes": int(n),
                                "% faltantes": round(100 * n / len(df), 3)})
nulos = pd.DataFrame(filas_nulos, columns=["Dataset", "columna", "faltantes", "% faltantes"])

# En pm25 cada año es una columna: se resume aparte para no llenar la tabla con 60+ años
anios_pm25 = [c for c in pm25.columns if str(c).strip().isdigit()]
nulos_sin_anios = nulos[~((nulos["Dataset"] == "pm25") & (nulos["columna"].isin(anios_pm25)))]
display(nulos_sin_anios)

anios_vacios = [c for c in anios_pm25 if pm25[c].isna().all()]
anios_con_dato = [c for c in anios_pm25 if pm25[c].notna().any()]
md(f"""
**pm25 (columnas de año):** {len(anios_pm25)} columnas de año ({anios_pm25[0]}–{anios_pm25[-1]}).
{len(anios_vacios)} están **completamente vacías**; hay datos solo entre **{anios_con_dato[0]} y {anios_con_dato[-1]}**.
""")

,Dataset,columna,faltantes,% faltantes
0,ev_registro,County,15,0.005
1,ev_registro,City,15,0.005
2,ev_registro,Postal Code,15,0.005
3,ev_registro,Electric Range,35,0.012
4,ev_registro,Legislative District,790,0.263
5,ev_registro,Vehicle Location,24,0.008
6,ev_registro,Electric Utility,15,0.005
7,ev_registro,2020 GEOID,15,0.005
8,ev_historial,County,117,0.317
9,ev_historial,State,117,0.317



**pm25 (columnas de año):** 66 columnas de año (1960–2025).
32 están **completamente vacías**; hay datos solo entre **1990 y 2023**.


### 4.2 Tipos de datos: ¿coherentes con lo que representan?

In [39]:
def tipo(df, col):
    return str(df[col].dtype) if col in df.columns else "columna no encontrada"


def es_numerico(df, col):
    return col in df.columns and pd.api.types.is_numeric_dtype(df[col])


COL_CENSO = next((c for c in ["2020 GEOID", "2020 Census Tract"] if c in ev_registro.columns), "2020 GEOID")

revision_tipos = [
    # (dataset, columna, tipo esperado, ¿coherente?, motivo)
    ("ev_registro", "Postal Code", "texto (código)", not es_numerico(ev_registro, "Postal Code"),
     "Es un código: no se suma ni promedia y un número pierde ceros a la izquierda."),
    ("ev_registro", "Legislative District", "texto o entero (categoría)",
     tipo(ev_registro, "Legislative District") != "float64",
     "Un distrito no puede ser 12.0; queda decimal porque hay faltantes."),
    ("ev_registro", COL_CENSO, "texto (código de 11 dígitos)",
     not es_numerico(ev_registro, COL_CENSO),
     "Código geográfico; como número se muestra en notación científica."),
    ("ev_registro", "Model Year", "entero", es_numerico(ev_registro, "Model Year"), "Año del modelo."),
    ("ev_registro", "Vehicle Location", "dos números (longitud, latitud)",
     es_numerico(ev_registro, "Vehicle Location"), "Viene como texto 'POINT (lon lat)'."),
    ("ev_historial", "Date", "fecha (datetime64)",
     pd.api.types.is_datetime64_any_dtype(ev_historial["Date"]) if "Date" in ev_historial else False,
     "Sin convertir no se puede ordenar ni agrupar por año."),
    ("ev_historial", "Electric Vehicle (EV) Total", "entero",
     es_numerico(ev_historial, "Electric Vehicle (EV) Total"), "Conteo de vehículos; si es texto, trae separador de miles."),
    ("ev_historial", "Total Vehicles", "entero",
     es_numerico(ev_historial, "Total Vehicles"), "Conteo de vehículos; si es texto, trae separador de miles."),
    ("ev_historial", "Percent Electric Vehicles", "decimal",
     es_numerico(ev_historial, "Percent Electric Vehicles"), "Porcentaje."),
    ("ev_ventas", "Year", "entero", es_numerico(ev_ventas, "Year"), "Año."),
    ("pm25", anios_con_dato[-1] if anios_con_dato else "(años)", "decimal en filas, no en columnas",
     False, "Los años son nombres de columna (texto); para analizar por año hay que pasar a formato largo."),
]

tipos_df = pd.DataFrame(
    [
        {"Dataset": d, "Columna": c, "Tipo actual": tipo(dataframes[d], c) if c in dataframes[d].columns else "—",
         "Tipo esperado": e, "¿Coherente?": "sí" if ok else "no", "Motivo": m}
        for d, c, e, ok, m in revision_tipos
        if c in dataframes[d].columns
    ]
)
tipos_df

,Dataset,Columna,Tipo actual,Tipo esperado,¿Coherente?,Motivo
0,ev_registro,Postal Code,float64,texto (código),no,Es un código: no se suma ni promedia y un núme...
1,ev_registro,Legislative District,float64,texto o entero (categoría),no,Un distrito no puede ser 12.0; queda decimal p...
2,ev_registro,2020 GEOID,float64,texto (código de 11 dígitos),no,Código geográfico; como número se muestra en n...
3,ev_registro,Model Year,int64,entero,sí,Año del modelo.
4,ev_registro,Vehicle Location,str,"dos números (longitud, latitud)",no,Viene como texto 'POINT (lon lat)'.
5,ev_historial,Date,str,fecha (datetime64),no,Sin convertir no se puede ordenar ni agrupar p...
6,ev_historial,Electric Vehicle (EV) Total,int64,entero,sí,"Conteo de vehículos; si es texto, trae separad..."
7,ev_historial,Total Vehicles,int64,entero,sí,"Conteo de vehículos; si es texto, trae separad..."
8,ev_historial,Percent Electric Vehicles,float64,decimal,sí,Porcentaje.
9,ev_ventas,Year,int64,entero,sí,Año.


### 4.3 Posibles inconsistencias

In [40]:
def numero(serie):
    """Convierte a número aunque venga como texto con separador de miles ('1,234')."""
    return pd.to_numeric(serie.astype(str).str.replace(",", "", regex=False), errors="coerce")


ANIO_ACTUAL = date.today().year
chequeos = {}

# ---------- Dataset 1: ev_registro
r = ev_registro
chequeos["ev_registro"] = {
    "filas duplicadas": int(r.duplicated().sum()),
    "DOL Vehicle ID repetido": int(r["DOL Vehicle ID"].duplicated().sum()) if "DOL Vehicle ID" in r else None,
    "Electric Range = 0": int((r["Electric Range"] == 0).sum()) if "Electric Range" in r else None,
    "% con Electric Range = 0": round(100 * (r["Electric Range"] == 0).mean(), 2) if "Electric Range" in r else None,
    "Base MSRP = 0": int((r["Base MSRP"] == 0).sum()) if "Base MSRP" in r else None,
    "State distinto de WA": int((r["State"] != "WA").sum()) if "State" in r else None,
    "Model Year mínimo": int(r["Model Year"].min()),
    "Model Year máximo": int(r["Model Year"].max()),
    f"Model Year > {ANIO_ACTUAL + 1}": int((r["Model Year"] > ANIO_ACTUAL + 1).sum()),
}

# ---------- Dataset 2: ev_historial
h = ev_historial
fechas = pd.to_datetime(h["Date"], errors="coerce", format="mixed")
bev, phev = numero(h["Battery Electric Vehicles (BEVs)"]), numero(h["Plug-In Hybrid Electric Vehicles (PHEVs)"])
ev_tot, no_ev = numero(h["Electric Vehicle (EV) Total"]), numero(h["Non-Electric Vehicle Total"])
total, porc = numero(h["Total Vehicles"]), numero(h["Percent Electric Vehicles"])
porc_recalc = 100 * ev_tot / total
claves_hist = [c for c in ["Date", "County", "State", "Vehicle Primary Use"] if c in h]
chequeos["ev_historial"] = {
    "fechas no convertibles": int(fechas.isna().sum()),
    "fecha mínima": str(fechas.min().date()),
    "fecha máxima": str(fechas.max().date()),
    "meses distintos": int(fechas.dt.to_period("M").nunique()),
    "claves repetidas (fecha, condado, estado, uso)": int(h.duplicated(subset=claves_hist).sum()),
    "BEV + PHEV ≠ total eléctrico": int(((bev + phev) != ev_tot).sum()),
    "eléctricos + no eléctricos ≠ total": int(((ev_tot + no_ev) != total).sum()),
    "% eléctrico no coincide al recalcular (>0,01)": int(((porc - porc_recalc).abs() > 0.01).sum()),
    "filas con estado distinto de WA": int((h["State"] != "WA").sum()) if "State" in h else None,
    "filas con Total Vehicles = 0": int((total == 0).sum()),
}

# ---------- Dataset 3: ev_ventas
v = ev_ventas
col_valor = [c for c in v.columns if c not in ("Entity", "Code", "Year")]
valor_ventas = v[col_valor[0]]
es_agregado = v["Code"].isna() | v["Code"].astype(str).str.startswith("OWID_")
chequeos["ev_ventas"] = {
    "columna de valor": col_valor[0],
    "entidades distintas": int(v["Entity"].nunique()),
    "entidades que son agregados (sin código ISO)": int(v.loc[es_agregado, "Entity"].nunique()),
    "año mínimo": int(v["Year"].min()),
    "año máximo": int(v["Year"].max()),
    "valores negativos": int((valor_ventas < 0).sum()),
    "valores iguales a 0": int((valor_ventas == 0).sum()),
    "pares (Entity, Year) repetidos": int(v.duplicated(subset=["Entity", "Year"]).sum()),
}

# ---------- Dataset 4: pm25
p = pm25
sin_nombre = [c for c in p.columns if str(c).startswith("Unnamed")]
paises_reales = set(pm25_paises.loc[pm25_paises["Region"].notna(), "Country Code"])
es_agregado_pm = ~p["Country Code"].isin(paises_reales)
valores_pm = p[anios_con_dato]
chequeos["pm25"] = {
    "columnas sin nombre (Unnamed)": len(sin_nombre),
    "columnas de año vacías": len(anios_vacios),
    "primer año con datos": anios_con_dato[0],
    "último año con datos": anios_con_dato[-1],
    "filas que son agregados (no países)": int(es_agregado_pm.sum()),
    "filas sin ningún valor de PM2.5": int(valores_pm.isna().all(axis=1).sum()),
    "valores negativos": int((valores_pm < 0).sum().sum()),
    "valor máximo (µg/m³)": round(float(valores_pm.max().max()), 2),
    "códigos de país repetidos": int(p["Country Code"].duplicated().sum()),
}

# Se quitan las comprobaciones de columnas que no existen en esta versión del archivo
chequeos = {n: {k: v for k, v in c.items() if v is not None} for n, c in chequeos.items()}

for nombre, resultado in chequeos.items():
    md(f"**{nombre}**")
    display(pd.DataFrame({"comprobación": list(resultado), "resultado": [valor(x, k) for k, x in resultado.items()]}))

**ev_registro**

,comprobación,resultado
0,filas duplicadas,0
1,DOL Vehicle ID repetido,0
2,Electric Range = 0,197.085
3,% con Electric Range = 0,65.66
4,State distinto de WA,789
5,Model Year mínimo,1999
6,Model Year máximo,2027
7,Model Year > 2027,0


**ev_historial**

,comprobación,resultado
0,fechas no convertibles,0
1,fecha mínima,2017-01-31
2,fecha máxima,2026-09-30
3,meses distintos,117
4,"claves repetidas (fecha, condado, estado, uso)",0
5,BEV + PHEV ≠ total eléctrico,0
6,eléctricos + no eléctricos ≠ total,0
7,"% eléctrico no coincide al recalcular (>0,01)",0
8,filas con estado distinto de WA,27.787
9,filas con Total Vehicles = 0,0


**ev_ventas**

,comprobación,resultado
0,columna de valor,Electric cars sold
1,entidades distintas,64
2,entidades que son agregados (sin código ISO),4
3,año mínimo,2010
4,año máximo,2025
5,valores negativos,4
6,valores iguales a 0,0
7,"pares (Entity, Year) repetidos",0


**pm25**

,comprobación,resultado
0,columnas sin nombre (Unnamed),1
1,columnas de año vacías,32
2,primer año con datos,1990
3,último año con datos,2023
4,filas que son agregados (no países),49
5,filas sin ningún valor de PM2.5,18
6,valores negativos,0
7,valor máximo (µg/m³),124.66
8,códigos de país repetidos,0


**Cómo leer estas comprobaciones**
- En `ev_registro`, `Electric Range = 0` (y `Base MSRP = 0`, en versiones del archivo que traen precio) **no** significan autonomía o precio cero: el Departamento de
  Licencias usa 0 cuando el dato no se ha investigado. Son faltantes disfrazados.
- En `ev_historial`, las sumas permiten validar que los totales del archivo son coherentes entre sí.
- En `ev_ventas` y `pm25`, los agregados (mundo, Europa, ingresos altos…) no son países: si se mezclan en un promedio
  se cuenta dos veces la misma población.

### 4.4 Limitaciones frente a la pregunta analítica (2021–2025)

In [41]:
PERIODO = set(range(2021, 2026))
cobertura = pd.DataFrame(
    [
        {"Dataset": "ev_registro", "Nivel geográfico": "vehículo → ciudad / condado (Washington)",
         "Años disponibles": f"modelos {chequeos['ev_registro']['Model Year mínimo']}–{chequeos['ev_registro']['Model Year máximo']} (no son años de registro)",
         "Años de 2021–2025 cubiertos": "no aplica (foto actual)"},
        {"Dataset": "ev_historial", "Nivel geográfico": "condado (Washington)",
         "Años disponibles": f"{fechas.min().year}–{fechas.max().year}",
         "Años de 2021–2025 cubiertos": len(PERIODO & set(fechas.dt.year.dropna().astype(int)))},
        {"Dataset": "ev_ventas", "Nivel geográfico": "país / región",
         "Años disponibles": f"{chequeos['ev_ventas']['año mínimo']}–{chequeos['ev_ventas']['año máximo']}",
         "Años de 2021–2025 cubiertos": len(PERIODO & set(ev_ventas["Year"].astype(int)))},
        {"Dataset": "pm25", "Nivel geográfico": "país / región",
         "Años disponibles": f"{anios_con_dato[0]}–{anios_con_dato[-1]}",
         "Años de 2021–2025 cubiertos": len(PERIODO & {int(a) for a in anios_con_dato})},
    ]
)
display(cobertura)

anios_pm_faltan = sorted(PERIODO - {int(a) for a in anios_con_dato})
pm_usa = {a: pm25.loc[pm25["Country Code"] == "USA", a].iloc[0] for a in anios_con_dato[-3:]} \
    if (pm25["Country Code"] == "USA").any() else {}
md(f"""
- **Escala distinta:** los vehículos eléctricos de Washington están por condado, pero el PM2.5 del Banco Mundial es un
  promedio **nacional** de EE. UU. No se pueden unir directamente condado con país.
- **Periodo incompleto:** al PM2.5 le faltan los años {", ".join(map(str, anios_pm_faltan)) or "—"} del periodo 2021–2025.
- **Dato modelado:** el PM2.5 del Banco Mundial combina satélites, modelos químicos y estaciones; no es una medición directa.
  Últimos valores de EE. UU.: {", ".join(f"{a}: {v}" for a, v in pm_usa.items()) or "no disponible"}.
- **Sin variables de clima:** la pregunta pide controlar lluvia y viento (NOAA), que todavía no están en el proyecto.
- **Ventas ≠ parque:** `ev_ventas` cuenta autos nuevos vendidos por año, no cuántos circulan.
""")

,Dataset,Nivel geográfico,Años disponibles,Años de 2021–2025 cubiertos
0,ev_registro,vehículo → ciudad / condado (Washington),modelos 1999–2027 (no son años de registro),no aplica (foto actual)
1,ev_historial,condado (Washington),2017–2026,5
2,ev_ventas,país / región,2010–2025,5
3,pm25,país / región,1990–2023,3



- **Escala distinta:** los vehículos eléctricos de Washington están por condado, pero el PM2.5 del Banco Mundial es un
  promedio **nacional** de EE. UU. No se pueden unir directamente condado con país.
- **Periodo incompleto:** al PM2.5 le faltan los años 2024, 2025 del periodo 2021–2025.
- **Dato modelado:** el PM2.5 del Banco Mundial combina satélites, modelos químicos y estaciones; no es una medición directa.
  Últimos valores de EE. UU.: 2021: 8.231389869812, 2022: 7.4282378905124, 2023: 6.96804491055748.
- **Sin variables de clima:** la pregunta pide controlar lluvia y viento (NOAA), que todavía no están en el proyecto.
- **Ventas ≠ parque:** `ev_ventas` cuenta autos nuevos vendidos por año, no cuántos circulan.


### 4.5 Tabla de hallazgos y acciones de limpieza propuestas

In [42]:
hallazgos = []


def agregar(dataset, problema, evidencia, impacto, accion, condicion=True):
    if condicion:
        hallazgos.append({"Dataset": dataset, "Problema identificado": problema,
                          "Evidencia encontrada": evidencia, "Posible impacto": impacto,
                          "Acción propuesta": accion})


cr, ch, cv, cp = (chequeos[k] for k in ["ev_registro", "ev_historial", "ev_ventas", "pm25"])

# ---- ev_registro
COLS_UBICACION = [c for c in ["County", "City", "Postal Code", "Electric Utility", COL_CENSO] if c in ev_registro]
sin_ubicacion = ev_registro[COLS_UBICACION].isna().all(axis=1)
estados_sin_ubic = ", ".join(f"{k} ({v})" for k, v in ev_registro.loc[sin_ubicacion, "State"].value_counts().head(4).items())
agregar("ev_registro", "Vehículos sin ubicación",
        f"{fmt(sin_ubicacion.sum())} filas con {', '.join(COLS_UBICACION)} vacíos a la vez; `State`: {estados_sin_ubic}",
        "No se pueden asignar a ningún condado de Washington.",
        "Excluirlos del análisis por condado (son registros con dirección fuera de EE. UU. o militar).",
        sin_ubicacion.sum() > 0)
IMPACTO_NULOS = {
    "Electric Range": ("Sin autonomía para esos modelos.", "Dejar como NaN; no imputar."),
    "Legislative District": ("No se pueden agrupar por distrito.", "No afecta el análisis por condado; dejar como NaN."),
    "Vehicle Location": ("No se pueden ubicar en un mapa.", "Usar `County` para el análisis; dejar como NaN."),
}
for _, f in nulos_sin_anios[nulos_sin_anios["Dataset"] == "ev_registro"].iterrows():
    if f["columna"] in COLS_UBICACION and f["faltantes"] <= sin_ubicacion.sum():
        continue  # ya explicado en "Vehículos sin ubicación"
    impacto, accion = IMPACTO_NULOS.get(f["columna"], ("Registros incompletos en esa variable.", "Revisar y dejar como NaN."))
    detalle = ""
    if f["columna"] == "Electric Range":
        anios = ev_registro.loc[ev_registro["Electric Range"].isna(), "Model Year"].value_counts().head(3)
        detalle = "; años de modelo: " + ", ".join(f"{k} ({v})" for k, v in anios.items())
    agregar("ev_registro", f"Faltantes en `{f['columna']}`",
            f"`isna().sum()` = {fmt(f['faltantes'])} ({pct(f['% faltantes'], 3)}){detalle}", impacto, accion)
agregar("ev_registro", "Autonomía en 0 usada como 'sin dato'", f"`Electric Range == 0`: {fmt(cr['Electric Range = 0'] or 0)} filas ({pct(cr['% con Electric Range = 0'] or 0)})",
        "El promedio de autonomía queda subestimado.", "Reemplazar 0 por NaN antes de calcular estadísticas.",
        (cr["Electric Range = 0"] or 0) > 0)
agregar("ev_registro", "Precio base en 0 usado como 'sin dato'", f"`Base MSRP == 0`: {fmt(cr.get('Base MSRP = 0') or 0)} filas",
        "La columna de precio casi no sirve para analizar.", "Reemplazar 0 por NaN o descartar la columna.",
        (cr.get("Base MSRP = 0") or 0) > 0)
agregar("ev_registro", "Vehículos registrados fuera de Washington", f"`State != 'WA'`: {fmt(cr['State distinto de WA'] or 0)} filas",
        "Inflan el conteo si se analiza solo Washington.", "Filtrar `State == 'WA'`.", (cr["State distinto de WA"] or 0) > 0)
agregar("ev_registro", "Códigos guardados como números",
        "`dtypes`: " + ", ".join(f"{c} = {tipo(ev_registro, c)}" for c in ["Postal Code", "Legislative District", COL_CENSO]),
        "Se pierden ceros a la izquierda y aparecen decimales (12.0) en códigos.",
        "Convertir a texto con `astype('Int64').astype(str)` o leer con `dtype=str`.")
agregar("ev_registro", "Ubicación como texto", "`Vehicle Location` contiene 'POINT (lon lat)'",
        "No se puede usar directamente en mapas.", "Extraer longitud y latitud con `str.extract()`.")
agregar("ev_registro", "Es una foto actual, no un historial", "No tiene columna de fecha de registro; `Model Year` es el año del modelo",
        "No sirve para medir el crecimiento año a año.", "Usar `ev_historial` para la evolución temporal.")
agregar("ev_registro", "Identificador repetido", f"`DOL Vehicle ID` duplicado: {fmt(cr['DOL Vehicle ID repetido'] or 0)}",
        "Un vehículo se contaría dos veces.", "Eliminar duplicados con `drop_duplicates('DOL Vehicle ID')`.",
        (cr["DOL Vehicle ID repetido"] or 0) > 0)

# ---- ev_historial
agregar("ev_historial", "Fecha guardada como texto", f"`dtype` de `Date` = {tipo(ev_historial, 'Date')}",
        "No se puede ordenar ni agrupar por año o mes.", "`pd.to_datetime(ev_historial['Date'])` y crear columna `anio`.",
        not pd.api.types.is_datetime64_any_dtype(ev_historial["Date"]))
nulos_hist = nulos_sin_anios[nulos_sin_anios["Dataset"] == "ev_historial"]
if len(nulos_hist):
    sin_ambos = int((ev_historial["County"].isna() & ev_historial["State"].isna()).sum())
    agregar("ev_historial", "Filas sin condado ni estado",
            "; ".join(f"`{f['columna']}`: {fmt(f['faltantes'])} ({pct(f['% faltantes'], 3)})" for _, f in nulos_hist.iterrows())
            + f"; {fmt(sin_ambos)} filas tienen ambos vacíos",
            "Vehículos que no se pueden asignar a un condado.", "Excluirlas de los totales por condado.")
cols_conteo = ["Battery Electric Vehicles (BEVs)", "Plug-In Hybrid Electric Vehicles (PHEVs)",
               "Electric Vehicle (EV) Total", "Non-Electric Vehicle Total", "Total Vehicles"]
conteos_texto = [c for c in cols_conteo if c in ev_historial and not es_numerico(ev_historial, c)]
con_coma = int(sum(ev_historial[c].astype(str).str.contains(",", regex=False).sum() for c in conteos_texto))
agregar("ev_historial", "Conteos leídos como texto",
        f"{len(conteos_texto)} columnas de conteo con dtype `str`; {fmt(con_coma)} valores con separador de miles (',')",
        "`describe()` no las resume y no se pueden sumar ni comparar.",
        "`pd.to_numeric(col.str.replace(',', ''))` en cada columna de conteo.", len(conteos_texto) > 0)
agregar("ev_historial", "Condados de otros estados", f"`State != 'WA'`: {fmt(ch['filas con estado distinto de WA'] or 0)} filas",
        f"Solo {pct(100 * (ev_historial['State'] == 'WA').mean())} de las filas son de Washington; el resto son condados donde viven dueños de vehículos registrados en WA.", "Filtrar `State == 'WA'`.",
        (ch["filas con estado distinto de WA"] or 0) > 0)
agregar("ev_historial", "Totales que no cuadran",
        f"BEV+PHEV≠total: {fmt(ch['BEV + PHEV ≠ total eléctrico'])}; EV+no EV≠total: {fmt(ch['eléctricos + no eléctricos ≠ total'])}",
        "Cifras poco confiables en esas filas.", "Recalcular los totales a partir de sus componentes.",
        ch["BEV + PHEV ≠ total eléctrico"] + ch["eléctricos + no eléctricos ≠ total"] > 0)
agregar("ev_historial", "Porcentaje que no coincide al recalcular",
        f"{fmt(ch['% eléctrico no coincide al recalcular (>0,01)'])} filas difieren de 100·EV/Total",
        "El % publicado y el calculado darían resultados distintos.", "Recalcular el porcentaje con 100 · EV / Total.",
        ch["% eléctrico no coincide al recalcular (>0,01)"] > 0)
agregar("ev_historial", "Dos tipos de uso en el mismo archivo",
        "`Vehicle Primary Use`: " + ", ".join(map(str, ev_historial["Vehicle Primary Use"].dropna().unique()[:4])),
        "Sumar todo mezcla pasajeros con camiones.", "Analizar `Passenger` por separado o sumar de forma explícita.")

# ---- ev_ventas
agregar("ev_ventas", "Mezcla de países y agregados", f"{fmt(cv['entidades que son agregados (sin código ISO)'])} entidades sin código ISO de país",
        "Promedios o sumas por país contarían dos veces.", "Filtrar filas con código ISO de 3 letras (excluir vacíos y `OWID_`).",
        cv["entidades que son agregados (sin código ISO)"] > 0)
agregar("ev_ventas", "Ventas negativas",
        "; ".join(f"{e} {a}: {fmt(x)}" for e, a, x in v.loc[valor_ventas < 0, ["Entity", "Year", cv["columna de valor"]]].itertuples(index=False)),
        "Una venta no puede ser negativa; es un residuo de calcular 'resto del mundo' por diferencia.",
        "Excluir el agregado 'Rest of World' o marcar esos valores como NaN.", cv["valores negativos"] > 0)
for _, f in nulos_sin_anios[nulos_sin_anios["Dataset"] == "ev_ventas"].iterrows():
    agregar("ev_ventas", f"Faltantes en `{f['columna']}`", f"`isna().sum()` = {fmt(f['faltantes'])} ({pct(f['% faltantes'])})",
            "Filas sin identificar como país.", "Tratarlas como agregados regionales.")
agregar("ev_ventas", "Nombre de columna largo", f"Columna de valor: '{cv['columna de valor']}'",
        "Código más difícil de leer.", "Renombrar a `ventas_ev`.", len(cv["columna de valor"]) > 15)
agregar("ev_ventas", "Ventas en cero", f"{fmt(cv['valores iguales a 0'])} filas con 0",
        "Puede ser dato real o falta de reporte en años tempranos.", "Verificar con la IEA; no imputar.", cv["valores iguales a 0"] > 0)

# ---- pm25
agregar("pm25", "Formato ancho (un año por columna)", f"{len(anios_pm25)} columnas de año",
        "No se puede unir por año con los otros datasets.", "Pasar a formato largo con `melt()` (país, año, pm25).")
agregar("pm25", "Columnas de año vacías", f"{cp['columnas de año vacías']} años sin ningún dato",
        "Ruido en la tabla.", "Eliminar esas columnas.", cp["columnas de año vacías"] > 0)
agregar("pm25", "Columna sin nombre", f"{cp['columnas sin nombre (Unnamed)']} columna(s) `Unnamed`",
        "Columna vacía creada por la coma final del CSV.", "Eliminarla.", cp["columnas sin nombre (Unnamed)"] > 0)
agregar("pm25", "Mezcla de países y agregados", f"{fmt(cp['filas que son agregados (no países)'])} filas no son países (según `Metadata_Country`)",
        "Doble conteo en comparaciones.", "Unir con los metadatos y conservar solo filas con `Region`.",
        cp["filas que son agregados (no países)"] > 0)
agregar("pm25", "Periodo incompleto", f"Último año con dato: {cp['último año con datos']}",
        "No cubre todo 2021–2025.", "Complementar con estaciones (EPA AirData u OpenAQ) para Washington.",
        bool(anios_pm_faltan))
agregar("pm25", "Escala nacional", "Un valor por país, no por condado",
        "No se puede relacionar directamente con los condados de Washington.",
        "Usar PM2.5 por condado (EPA AirData) para el análisis local; dejar este dataset como contexto.")

hallazgos_df = pd.DataFrame(hallazgos)
display(hallazgos_df)
md(f"Se documentan **{len(hallazgos_df)} hallazgos** en {hallazgos_df['Dataset'].nunique()} datasets.")

,Dataset,Problema identificado,Evidencia encontrada,Posible impacto,Acción propuesta
0,ev_registro,Vehículos sin ubicación,"15 filas con County, City, Postal Code, Electr...",No se pueden asignar a ningún condado de Washi...,Excluirlos del análisis por condado (son regis...
1,ev_registro,Faltantes en `Electric Range`,"`isna().sum()` = 35 (0,012 %); años de modelo:...",Sin autonomía para esos modelos.,Dejar como NaN; no imputar.
2,ev_registro,Faltantes en `Legislative District`,"`isna().sum()` = 790 (0,263 %)",No se pueden agrupar por distrito.,No afecta el análisis por condado; dejar como ...
3,ev_registro,Faltantes en `Vehicle Location`,"`isna().sum()` = 24 (0,008 %)",No se pueden ubicar en un mapa.,Usar `County` para el análisis; dejar como NaN.
4,ev_registro,Autonomía en 0 usada como 'sin dato',"`Electric Range == 0`: 197.085 filas (65,66 %)",El promedio de autonomía queda subestimado.,Reemplazar 0 por NaN antes de calcular estadís...
5,ev_registro,Vehículos registrados fuera de Washington,`State != 'WA'`: 789 filas,Inflan el conteo si se analiza solo Washington.,Filtrar `State == 'WA'`.
6,ev_registro,Códigos guardados como números,"`dtypes`: Postal Code = float64, Legislative D...",Se pierden ceros a la izquierda y aparecen dec...,Convertir a texto con `astype('Int64').astype(...
7,ev_registro,Ubicación como texto,`Vehicle Location` contiene 'POINT (lon lat)',No se puede usar directamente en mapas.,Extraer longitud y latitud con `str.extract()`.
8,ev_registro,"Es una foto actual, no un historial",No tiene columna de fecha de registro; `Model ...,No sirve para medir el crecimiento año a año.,Usar `ev_historial` para la evolución temporal.
9,ev_historial,Fecha guardada como texto,`dtype` de `Date` = str,No se puede ordenar ni agrupar por año o mes.,`pd.to_datetime(ev_historial['Date'])` y crear...


Se documentan **23 hallazgos** en 4 datasets.

---
## Momento 5. Elaboración del informe
La siguiente celda escribe `reports/informe_diagnostico.md` con las tablas y cifras calculadas arriba.
Si se vuelven a descargar los datos y se ejecuta el notebook, el informe se actualiza solo.

In [43]:
RUTA_INFORME = Path("../reports/informe_diagnostico.md")
if not RUTA_INFORME.parent.exists():
    RUTA_INFORME = Path("reports/informe_diagnostico.md")

ver_cols = ["Dataset", "columna", "faltantes", "% faltantes"]
total_faltantes = nulos_sin_anios.copy()
total_faltantes["% faltantes"] = total_faltantes["% faltantes"].map(lambda x: pct(x, 3))
total_faltantes["faltantes"] = total_faltantes["faltantes"].map(fmt)

resumen_chequeos = "\n\n".join(
    f"**{n}**\n\n" + tabla_md(pd.DataFrame({"comprobación": list(c.keys()), "resultado": [valor(x, k) for k, x in c.items()]}))
    for n, c in chequeos.items()
)

hallazgos_por_ds = hallazgos_df.groupby("Dataset", sort=False)["Problema identificado"].apply(list)
texto_hallazgos = "\n".join(
    f"- **{ds}:** " + "; ".join(p.replace("`", "") for p in probs) + "."
    for ds, probs in hallazgos_por_ds.items()
)

informe = f"""# Informe técnico · Diagnóstico inicial de calidad de datos

**Actividad:** GA1-220501094-01-AA1-EV06
**Aprendiz:** Juan Camilo Sarrazola Agudelo · Ficha 3169901 · SENA
**Fecha de ejecución del diagnóstico:** {date.today().isoformat()}
**Generado por:** `notebooks/diagnostico_calidad.ipynb` (todas las cifras salen de Pandas)

## 1. Proyecto y pregunta analítica
**Proyecto:** Vehículos eléctricos y calidad del aire: ¿el aumento de carros eléctricos ayuda a reducir el PM2.5 en las ciudades?

**Pregunta analítica:** ¿El aumento de vehículos eléctricos en las ciudades se relaciona con una baja del PM2.5 (µg/m³) entre 2021 y 2025?

## 2. Datasets utilizados
| DataFrame | Archivo | Fuente | Cada fila es |
|---|---|---|---|
| ev_registro | {archivos['ev_registro']} | Data.gov · Departamento de Licencias de Washington | {QUE_ES_UNA_FILA['ev_registro']} |
| ev_historial | {archivos['ev_historial']} | Data.gov · Departamento de Licencias de Washington | {QUE_ES_UNA_FILA['ev_historial']} |
| ev_ventas | {archivos['ev_ventas']} | Our World in Data (IEA) | {QUE_ES_UNA_FILA['ev_ventas']} |
| pm25 | {archivos['pm25']} | Banco Mundial (EN.ATM.PM25.MC.M3) | {QUE_ES_UNA_FILA['pm25']} |

## 3. Resultados de la inspección estructural
Se aplicaron `head()`, `tail()`, `shape`, `columns`, `dtypes`, `info()`, `describe()` y `describe(include='object')` a cada DataFrame.

{tabla_md(estructura)}

## 4. Valores faltantes, tipos de datos e inconsistencias
### 4.1 Valores faltantes (sin contar las columnas de año de pm25)
{tabla_md(total_faltantes[ver_cols]) if len(total_faltantes) else "No se encontraron valores faltantes."}

En `pm25` hay {len(anios_pm25)} columnas de año; {len(anios_vacios)} están vacías y solo hay datos entre {anios_con_dato[0]} y {anios_con_dato[-1]}.

### 4.2 Tipos de datos
{tabla_md(tipos_df)}

### 4.3 Comprobaciones de consistencia
{resumen_chequeos}

## 5. Principales hallazgos de calidad por dataset
{texto_hallazgos}

## 6. Acciones de limpieza y preparación propuestas
{tabla_md(hallazgos_df)}

## 7. Conclusiones
1. Los cuatro datasets cargan sin errores y tienen estructura tabular clara, pero **ninguno está listo para el análisis tal como viene**:
   hay tipos incoherentes (fechas y códigos), ceros que significan "sin dato" y agregados mezclados con países.
   Los problemas de mayor peso son: {pct(chequeos['ev_registro'].get('% con Electric Range = 0', 0))} de autonomías registradas en 0 en `ev_registro`,
   {pct(100 * (ev_historial['State'] != 'WA').mean())} de filas de `ev_historial` que no son de Washington, los conteos del historial leídos como texto
   y {chequeos['ev_ventas']['valores negativos']} ventas negativas en `ev_ventas`.
2. La limitación más importante no es de limpieza sino de **cobertura**: los vehículos eléctricos están por condado de Washington
   y el PM2.5 por país, y al PM2.5 le faltan {len(anios_pm_faltan)} año(s) del periodo 2021–2025 ({", ".join(map(str, anios_pm_faltan)) or "ninguno"}).
3. Para responder la pregunta a nivel local se propone agregar PM2.5 por condado (EPA AirData u OpenAQ) y datos de clima (NOAA).
   `ev_ventas` y `pm25` quedan como contexto internacional.
4. Con las acciones propuestas (conversión de tipos, filtro a Washington, exclusión de agregados, formato largo para pm25)
   los datos quedan en condiciones de pasar a la fase de limpieza y unión.
"""

RUTA_INFORME.write_text(informe, encoding="utf-8")
md(f"Informe escrito en `{RUTA_INFORME}` ({len(informe):,} caracteres).".replace(",", "."))

Informe escrito en `..\reports\informe_diagnostico.md` (12.496 caracteres).

---
## Momento 6. Verificación antes de publicar

In [44]:
RAIZ = RUTA_INFORME.parent.parent
gitignore = (RAIZ / ".gitignore").read_text(encoding="utf-8") if (RAIZ / ".gitignore").exists() else ""

verificaciones = {
    "Los 4 datasets están en la tabla estructural": set(estructura["Dataset"]) == set(dataframes),
    "Los 4 datasets tienen hallazgos documentados": set(hallazgos_df["Dataset"]) == set(dataframes),
    "Existe reports/informe_diagnostico.md": RUTA_INFORME.exists(),
    "Existe README.md": (RAIZ / "README.md").exists(),
    "Existe requirements.txt": (RAIZ / "requirements.txt").exists(),
    "Los CSV están excluidos en .gitignore": "data/raw/*.csv" in gitignore,
    "Existe notebooks/diagnostico_calidad.ipynb": (RAIZ / "notebooks" / "diagnostico_calidad.ipynb").exists(),
}
for texto, ok in verificaciones.items():
    print("✔" if ok else "✘", texto)

✔ Los 4 datasets están en la tabla estructural
✔ Los 4 datasets tienen hallazgos documentados
✔ Existe reports/informe_diagnostico.md
✔ Existe README.md
✘ Existe requirements.txt
✔ Los CSV están excluidos en .gitignore
✔ Existe notebooks/diagnostico_calidad.ipynb
